# 🚀 Colab LLM Coding Server (vLLM + Cloudflare Tunnel)

This notebook turns your Google Colab instance (A100 or L4 GPU) into an **OpenAI-compatible Coding API**.
You can connect **Aider**, **Continue.dev**, or our custom **Claude Code-style CLI** directly to this endpoint!

### Recommended Colab Settings:
* **Runtime** -> **Change runtime type** -> **A100 GPU** (or L4) + **High-RAM**.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

In [ ]:
import sys
# Uninstall existing torchaudio if present to prevent conflicts
!pip uninstall -y torchaudio > /dev/null 2>&1
# Step 2: Install vLLM and Cloudflared
!pip install -q -U vllm
# Force reinstall torchaudio for CUDA 13.0 to match PyTorch CUDA
!pip install -q -U --force-reinstall torchaudio --index-url https://download.pytorch.org/whl/cu130
# Install cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


In [ ]:
# Step 3: Choose your model
# Qwen2.5-Coder-32B-Instruct-AWQ is quantized 4-bit, runs super fast, and easily fits in A100/L4 VRAM!
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ"
MAX_MODEL_LEN = 16384  # Adjust up to 32768 if needed

print(f"Selected Model: {MODEL_NAME}")

In [ ]:
# Step 4: Launch vLLM Server & Cloudflare Tunnel
import subprocess
import time
import re
import sys

# Start vLLM in the background
print("Starting vLLM server...")
vllm_cmd = [
    "vllm", "serve",
    MODEL_NAME,
    "--port", "8000",
    "--host", "0.0.0.0",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
    "--trust-remote-code"
]
vllm_proc = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Start Cloudflare Tunnel
print("Creating Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Capture the public URL
tunnel_url = None
while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "="*60)
print("🎉 TUNNEL ACTIVE!")
print(f"API Base URL: {tunnel_url}/v1")
print("="*60 + "\n")
print("Copy this URL and paste it into your local coding CLI or Aider!")

# Stream vLLM logs so you can see when the model finishes loading
while True:
    v_line = vllm_proc.stdout.readline()
    if v_line:
        print(v_line, end="")
    time.sleep(0.01)